# 식수 인원 예측 성능 향상 (V3)

3위 수상작인 `XGBOOST와 원초적 본능.ipynb`의 전략을 반영하여, XGBoost 모델 사용, 석식 이상치(자기개발의 날 등) 제거, 그리고 메뉴 파싱 단순화를 적용한 모델입니다.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

import korean_font
korean_font.set_korean_font()

한글 폰트 설정: Malgun Gothic (c:/Windows/Fonts/malgun.ttf)


## 1. 데이터 로드

In [3]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
submission = pd.read_csv('data/sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1205, 12)
Test shape: (50, 10)


## 2. 데이터 전처리

### 2.1 날짜 및 인원 파생 변수
- 3위 전략: Month, Day를 정수형으로 사용
- 사내근무자(in_office) 계산

In [ ]:
def process_date(df):
    df['일자'] = pd.to_datetime(df['일자'])
    df['month'] = df['일자'].dt.month
    df['day'] = df['일자'].dt.day
    df['weekday'] = df['일자'].dt.weekday
    return df

train = process_date(train)
test = process_date(test)

def process_personnel(df):
    # 사내근무자 (in_office)
    df['in_office'] = df['본사정원수'] - df['본사휴가자수'] - df['본사출장자수'] - df['현본사소속재택근무자수']
    return df
```````
train = process_personnel(train)
test = process_personnel(test)

### 2.2 메뉴 파싱 (단순화)
- 복잡도를 낮추기 위해 '밥', '국', '메인' 3가지만 추출합니다.
- '(' 가 포함된 원산지 정보는 삭제합니다.

In [5]:
def parse_menu(menu_series):
    bob = []
    soup = []
    main = []
    for menu_str in menu_series:
        if pd.isna(menu_str) or menu_str.strip() == '':
            bob.append('None')
            soup.append('None')
            main.append('None')
            continue
            
        items = menu_str.split()
        # 괄호 제거 로직
        clean_items = []
        for item in items:
            if '(' in item:
                continue
            clean_items.append(item)
        
        b = clean_items[0] if len(clean_items) > 0 else 'None'
        s = clean_items[1] if len(clean_items) > 1 else 'None'
        m = clean_items[2] if len(clean_items) > 2 else 'None'
        
        bob.append(b)
        soup.append(s)
        main.append(m)
    return bob, soup, main

train['lunch_bob'], train['lunch_soup'], train['lunch_main'] = parse_menu(train['중식메뉴'])
train['dinner_bob'], train['dinner_soup'], train['dinner_main'] = parse_menu(train['석식메뉴'])

test['lunch_bob'], test['lunch_soup'], test['lunch_main'] = parse_menu(test['중식메뉴'])
test['dinner_bob'], test['dinner_soup'], test['dinner_main'] = parse_menu(test['석식메뉴'])

### 2.3 석식 데이터 정제 (이상치 제거)
- '자기개발의 날' 등 석식 제공을 하지 않거나 인원이 극히 적은 날들을 학습에서 제외합니다.

In [6]:
# 석식 메뉴가 없거나 '자기개발의날' 등으로 표시된 경우 제외
# 3위 수상작의 조건 반영
train_dinner = train.copy()
train_dinner = train_dinner[train_dinner['석식메뉴'].str.contains('\*', na=False) == False]
train_dinner = train_dinner[train_dinner['dinner_bob'] != 'None']
train_dinner = train_dinner[train_dinner['석식계'] > 0]

print("Original train shape:", train.shape)
print("Cleaned dinner train shape:", train_dinner.shape)

Original train shape: (1205, 22)
Cleaned dinner train shape: (959, 22)


### 2.4 범주형 데이터 인코딩
- `LabelEncoder`를 사용하며, Test의 미학습 레이블은 'Unknown'으로 처리한 후 0으로 매핑합니다.

In [7]:
def encode_features(train_df, test_df, cat_cols):
    for col in cat_cols:
        le = LabelEncoder()
        # Train 데이터 기준으로 학습
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        
        # Test 데이터 인코딩 (미학습 레이블 처리)
        mapping = {l: i for i, l in enumerate(le.classes_)}
        test_df[col] = test_df[col].astype(str).apply(lambda x: mapping.get(x, -1))
    return train_df, test_df

cat_cols = ['lunch_bob', 'lunch_soup', 'lunch_main', 'dinner_bob', 'dinner_soup', 'dinner_main']

# 중식용 데이터 인코딩
train_lunch_encoded, test_lunch_encoded = encode_features(train.copy(), test.copy(), cat_cols)
# 석식용 데이터 인코딩 (정제된 train_dinner 사용)
train_dinner_encoded, test_dinner_encoded = encode_features(train_dinner.copy(), test.copy(), cat_cols)

## 3. 모델링 (XGBoost)

### 3.1 하이퍼파라미터 설정
- Reference에서 사용된 최적 파라미터 적용 가이드:
    - `max_depth = 3`
    - `n_estimators = 600`
    - `learning_rate = 0.1`
    - `colsample_bytree = 0.7`
    - `colsample_bylevel = 0.5`

In [8]:
features_lunch = ['weekday', 'month', 'day', 'in_office', '본사시간외근무명령서승인건수', 'lunch_bob', 'lunch_soup', 'lunch_main']
features_dinner = ['weekday', 'month', 'day', 'in_office', '본사시간외근무명령서승인건수', 'dinner_bob', 'dinner_soup', 'dinner_main']

def train_xgb(X, y, test_X, target_name):
    n_splits = 5
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(test_X))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = XGBRegressor(
            n_estimators=600,
            max_depth=3,
            learning_rate=0.1,
            colsample_bytree=0.7,
            colsample_bylevel=0.5,
            random_state=42,
            verbosity=0,
            early_stopping_rounds=50
        )
        
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        
        val_pred = model.predict(X_va)
        oof_preds[val_idx] = val_pred
        test_preds += model.predict(test_X) / n_splits
        
        mae = mean_absolute_error(y_va, val_pred)
        print(f"{target_name} Fold {fold+1} MAE: {mae:.4f}")
        
    print(f"{target_name} Total OOF MAE: {mean_absolute_error(y, oof_preds):.4f}\n")
    return test_preds

print("--- 중식계 모델 학습 ---")
target_lunch_col = train.columns[10] # '중식계'
preds_lunch = train_xgb(train_lunch_encoded[features_lunch], train_lunch_encoded[target_lunch_col], test_lunch_encoded[features_lunch], "중식계")

print("--- 석식계 모델 학습 ---")
target_dinner_col = train.columns[11] # '석식계'
preds_dinner = train_xgb(train_dinner_encoded[features_dinner], train_dinner_encoded[target_dinner_col], test_dinner_encoded[features_dinner], "석식계")

--- 중식계 모델 학습 ---
중식계 Fold 1 MAE: 83.0206
중식계 Fold 2 MAE: 77.6659
중식계 Fold 3 MAE: 72.8555
중식계 Fold 4 MAE: 89.9306
중식계 Fold 5 MAE: 88.3595
중식계 Total OOF MAE: 82.3664

--- 석식계 모델 학습 ---
석식계 Fold 1 MAE: 49.7273
석식계 Fold 2 MAE: 55.8939
석식계 Fold 3 MAE: 55.0646
석식계 Fold 4 MAE: 53.2696
석식계 Fold 5 MAE: 60.6847
석식계 Total OOF MAE: 54.9220



## 4. 최종 예측 리포트 및 저장

In [9]:
submission['중식계'] = preds_lunch
submission['석식계'] = preds_dinner

# 결과 저장
submission.to_csv('submission/dining_submission_v3.csv', index=False)
print("제출 파일이 submission/dining_submission_v3.csv 에 저장되었습니다.")
display(submission.head())

제출 파일이 submission/dining_submission_v3.csv 에 저장되었습니다.


,일자,중식계,석식계
0,2021-01-27,934.484299,326.478802
1,2021-01-28,885.741226,478.159050
2,2021-01-29,741.905289,226.504639
3,2021-02-01,1257.303589,520.854996
4,2021-02-02,1096.701355,520.450729
